# Session 09: LangGraph Platform

## Deploying the Stone Ridge Investment Assistant with LangGraph Platform

### Learning Objectives

- **Understand LangGraph Platform architecture** — how `langgraph.json`, `app/agent.py`, and the CLI work together
- **Walk through the consolidated agent** — understand the graph structure with guardrails, helpfulness evaluation, and investment-domain tools
- **Deploy locally with `langgraph dev`** — run the investment assistant as a local API server
- **Test via the LangGraph SDK** — interact with the deployed agent programmatically

### Overview

In this notebook we explore the **`app/agent.py`** file — a consolidated LangGraph agent that combines:
- **Anthropic Claude** as the LLM (Claude Sonnet for main agent, Claude Haiku for evaluation)
- **RAG** over the Stone Ridge 2025 Investor Letter
- **X/Twitter tools** for market sentiment analysis
- **Tavily + Arxiv** for web and academic search
- **Input/output guardrails** for production safety
- **Helpfulness evaluation** loop for response quality

The agent is deployed via **LangGraph Platform**, which provides:
- A REST API for invocation
- Built-in streaming support
- Thread-based memory management
- LangSmith integration for observability

---

## Task 1: Understand the Agent Architecture

Let's start by examining the key components of our consolidated agent.

### Graph Architecture

```
START \u2192 input_guardrail \u2192[pass]\u2192 agent \u2192[tools?]\u2192 action \u2192 agent (loop)
                         [fail]\u2192 END        [no tools]\u2192 output_guardrail \u2192[pass]\u2192 helpfulness \u2192[Y]\u2192 END
                                                                          [fail]\u2192 agent (retry)   [N]\u2192 agent
```

### Key Components

| Component | Description | Model |
|-----------|-------------|-------|
| Input Guardrail | Topic restriction, jailbreak detection, PII protection | Guardrails AI |
| Agent | Main reasoning + tool calling | Claude Sonnet |
| Action | Tool execution (Tavily, Arxiv, RAG, X/Twitter) | N/A |
| Output Guardrail | PII protection, profanity filter | Guardrails AI |
| Helpfulness | Response quality evaluation | Claude Haiku |

In [ ]:
import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key (for embeddings):")

# Optional
if not os.environ.get("TAVILY_API_KEY"):
    tavily = getpass.getpass("Tavily API Key (optional \u2014 Enter to skip):")
    if tavily.strip():
        os.environ["TAVILY_API_KEY"] = tavily

### Examining the Agent Code

Let's look at the key sections of `app/agent.py`:

In [ ]:
# View the agent module structure
with open("app/agent.py") as f:
    lines = f.readlines()

# Print section headers
for i, line in enumerate(lines, 1):
    if line.startswith("# ----") or line.startswith("def ") or line.startswith("class "):
        print(f"{i:4d}: {line.rstrip()}")

---

## Task 2: Import and Compile the Graph Locally

Before deploying to LangGraph Platform, let's import and test the graph directly.

In [ ]:
from app.agent import (
    graph,
    build_graph,
    get_chat_model,
    get_tool_belt,
    SYSTEM_PROMPT,
    DEFAULT_MODEL,
    EVAL_MODEL,
)

print(f"Graph compiled successfully!")
print(f"Default model: {DEFAULT_MODEL}")
print(f"Eval model: {EVAL_MODEL}")
print(f"\nTools available:")
for t in get_tool_belt():
    print(f"  - {t.name}")

In [ ]:
# Visualize the graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

---

## Task 3: Test the Graph Locally

Let's invoke the graph directly with some investment-related queries.

In [ ]:
from langchain_core.messages import HumanMessage

# Test with an investment question
result = graph.invoke(
    {"messages": [HumanMessage(content="What is Stone Ridge's investment philosophy regarding reinsurance?")]}
)

# Print the final response (skip HELPFULNESS markers)
for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and not msg.content.startswith("HELPFULNESS:"):
        print(msg.content)
        break

In [ ]:
# Test the guardrails with an off-topic query
result = graph.invoke(
    {"messages": [HumanMessage(content="What medicine should I take for a headache?")]}
)

for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and msg.content:
        print(msg.content)
        break

In [ ]:
# Test with a query that should use the RAG tool
result = graph.invoke(
    {"messages": [HumanMessage(
        content="According to the Stone Ridge 2025 Investor Letter, how does the firm think about bitcoin allocation?"
    )]}
)

for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and not msg.content.startswith("HELPFULNESS:"):
        print(msg.content)
        break

---

## Task 4: Deploy with LangGraph Platform

Now let's deploy the agent using LangGraph Platform. The configuration is defined in `langgraph.json`:

```json
{
  "version": 1,
  "dependencies": ["."],
  "env": ".env",
  "python_version": "3.13",
  "graphs": {
    "investment_assistant": "app.agent:graph"
  }
}
```

### Starting the Server

In a terminal, run:

```bash
cd 09_Production_and_MCP
uv run langgraph dev
```

This will:
1. Install dependencies from `pyproject.toml`
2. Load environment variables from `.env`
3. Start a local API server at `http://localhost:2024`
4. Open the LangGraph Studio UI for visual debugging

---

## Task 5: Test via LangGraph SDK

Once the server is running, we can interact with it using the **LangGraph SDK**. This is how production clients would call the agent.

In [ ]:
from langgraph_sdk import get_sync_client

# Connect to the local LangGraph Platform server
client = get_sync_client(url="http://localhost:2024")

print("Connected to LangGraph Platform!")

# List available assistants
assistants = client.assistants.search()
for a in assistants:
    print(f"  - {a['assistant_id']}: {a.get('name', 'unnamed')}")

In [ ]:
# Stream a response from the deployed agent
for chunk in client.runs.stream(
    None,  # Threadless run
    "investment_assistant",
    input={
        "messages": [
            {
                "role": "human",
                "content": "What is Stone Ridge's view on reinsurance as an asset class?",
            }
        ]
    },
    stream_mode="updates",
):
    print(f"Event: {chunk.event}")
    if chunk.data:
        # Print the last message content if available
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(f"  {content[:200]}..." if len(content) > 200 else f"  {content}")
    print()

In [ ]:
# Test with a thread for multi-turn conversation
thread = client.threads.create()
print(f"Thread created: {thread['thread_id']}")

# First message
for chunk in client.runs.stream(
    thread["thread_id"],
    "investment_assistant",
    input={"messages": [{"role": "human", "content": "Tell me about Stone Ridge's bitcoin strategy."}]},
    stream_mode="updates",
):
    if chunk.event == "updates" and chunk.data:
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(content[:500])

In [ ]:
# Follow-up question (same thread = agent remembers context)
for chunk in client.runs.stream(
    thread["thread_id"],
    "investment_assistant",
    input={"messages": [{"role": "human", "content": "How does that compare to their reinsurance approach?"}]},
    stream_mode="updates",
):
    if chunk.event == "updates" and chunk.data:
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(content[:500])

### \u2753 Question #1

What are the benefits of deploying an agent through LangGraph Platform versus running it directly as a Python script? Consider: scalability, observability, memory management, and client access.

##### Answer

*Your answer here*

### \u2753 Question #2

How does the helpfulness evaluation loop affect latency and cost? When would you enable or disable it in production?

##### Answer

*Your answer here*

### \ud83c\udfd7\ufe0f Activity #1

Modify the agent to add a new graph variant:

1. Create a "simple" variant in `app/agent.py` that skips guardrails and helpfulness (just agent + action nodes)
2. Register it in `langgraph.json` as `"simple_assistant"`
3. Restart `langgraph dev` and test both variants via the SDK
4. Compare response times and quality

In [ ]:
### YOUR CODE HERE

---

## Summary

In this notebook we covered:

- **Agent Architecture**: Walked through the consolidated `app/agent.py` with guardrails, tools, and helpfulness evaluation
- **Local Testing**: Imported and tested the graph directly in Python
- **LangGraph Platform Deployment**: Used `langgraph dev` to deploy as a local API server
- **SDK Client**: Tested the deployed agent with streaming and multi-turn threads

### Key Takeaways

1. **LangGraph Platform = deployment made simple** \u2014 `langgraph.json` + `app/agent.py` is all you need
2. **The SDK provides production-grade access** \u2014 threads, streaming, and async are built in
3. **Guardrails add safety but also latency** \u2014 consider the tradeoff per use case
4. **The helpfulness loop improves quality** \u2014 but costs an extra LLM call per response